# Taller 6 · Filtrado digital: sistemas, ecuaciones en diferencias y segmentación

**Asignatura:** Teoría de la Información y Procesado de Señal  
**Grado en Ciencia e Ingeniería de Datos (GCED) — Universidad de A Coruña**  
**Duración:** 2 horas  
**Modalidad:** Jupyter Notebook con asistencia de IA

---

## Objetivos de aprendizaje

Al finalizar este taller serás capaz de:

1. Implementar **filtros FIR** básicos (media móvil, diferenciador) y observar su efecto.
2. **Programar desde cero** la ecuación en diferencias general que describe un sistema LTI, con coeficientes $b$ y $a$.
3. Comparar tu implementación con `scipy.signal.lfilter()`.
4. Aplicar sistemas **FIR e IIR** a señales concretas (filtrado de tonos).
5. Entender el **filtrado por etapas** (segmentación) y la importancia de las **condiciones iniciales/finales**.
6. Calcular e interpretar la **autocorrelación** y la **correlación cruzada**.

---

## Reto central del taller

> **Implementar, aplicar y segmentar — todo en el dominio del tiempo.**
>
> En este taller programarás tu propia función de ecuaciones en diferencias, la aplicarás a señales reales, y descubrirás por qué no se puede filtrar "a trozos" sin propagación de estado.
>
> **En el Taller 9 usaremos exactamente las mismas señales y sistemas, pero vistos desde la frecuencia.** Allí entenderás *por qué* pasa lo que aquí observas.

---

## Ecuación general del sistema

Todo el taller gira en torno a la ecuación en diferencias vista en clase de teoría:

$$\sum_{l=0}^{L} a_l\, y(n-l) = \sum_{m=0}^{M} b_m\, x(n-m) \qquad [1]$$

que, despejando $y(n)$, queda:

$$y(n) = \frac{1}{a_0}\left(\sum_{m=0}^{M} b_m\, x(n-m) - \sum_{l=1}^{L} a_l\, y(n-l)\right)$$

donde:
- $x(n)$ es la señal de entrada, $y(n)$ la señal de salida.
- $b_m$ ($M+1$ coeficientes) y $a_l$ ($L+1$ coeficientes) son los parámetros del sistema.
- Si todos los $a_l = 0$ para $l \geq 1$ → sistema **FIR** (solo depende de la entrada).
- Si algún $a_l \neq 0$ para $l \geq 1$ → sistema **IIR** (depende también de salidas pasadas → recursividad).
- **Condiciones iniciales**: se suelen asumir nulas, $x(n)=0$ e $y(n)=0$ para $n<0$.

---

## Metodología de trabajo con IA

| Puedes pedir a la IA | NO debes pedir a la IA |
|---------------------|------------------------|
| Sintaxis de `lfilter`, `np.convolve`, `signal.correlate` | Que interprete los resultados |
| Depuración de errores en tu código | Que complete las explicaciones |
| Dudas sobre indexación de arrays | Que responda las preguntas de control |

> *La IA te ayuda a escribir código, pero no a entender sistemas. Eso es tu trabajo.*

---

## Identificación del estudiante

Completa los siguientes campos con tu información personal:

- **Apellidos:** _____

- **Nombre:** _____

- **Email UDC:** _____

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

np.random.seed(42)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

print("✓ Entorno listo")

---

## Parte 0: Generación de las señales base

Estas señales se usarán en todo el taller.

In [ ]:
# === PARÁMETROS COMUNES (usados en T6 y T9) ===
N = 500          # número de muestras
n = np.arange(N)

# Frecuencias normalizadas (ciclos por muestra)
f1 = 0.05   # frecuencia baja
f2 = 0.30   # frecuencia alta

# Orden del filtro de media móvil
M_media = 10

# === SEÑAL x1: suma de dos tonos ===
x1a = np.sin(2 * np.pi * f1 * n)
x1b = np.sin(2 * np.pi * f2 * n)
x1 = x1a + x1b

# === SEÑAL x2: ruido blanco (distribución gaussiana) ===
x2 = np.random.randn(N)

# === FILTRO h1: media móvil ===
h_media = np.ones(M_media) / M_media

# === SEÑAL x3: ruido suavizado ===
x3 = np.convolve(x2, h_media, mode='same')

print(f"✓ Señales generadas:")
print(f"  x1: suma de tonos (f1={f1}, f2={f2})")
print(f"  x2: ruido blanco gaussiano")
print(f"  x3: ruido suavizado (M={M_media})")

In [ ]:
# === VISUALIZACIÓN DE LAS TRES SEÑALES ===
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(n, x1, 'b-', linewidth=1)
axes[0].set_title(f'$x_1[n]$: suma de tonos ($f_1$={f1}, $f_2$={f2})')
axes[0].set_ylabel('Amplitud')

axes[1].plot(n, x2, 'orange', linewidth=0.8, alpha=0.8)
axes[1].set_title('$x_2[n]$: ruido blanco')
axes[1].set_ylabel('Amplitud')

axes[2].plot(n, x3, 'green', linewidth=1)
axes[2].set_title(f'$x_3[n]$: ruido suavizado (media móvil M={M_media})')
axes[2].set_xlabel('n (muestras)')
axes[2].set_ylabel('Amplitud')

plt.tight_layout()
plt.show()

---

## Parte 1: Sistemas FIR en el dominio del tiempo

### Bloque 1.1 · ¿Qué hace el filtro de media móvil del apartado anterior, $h_{media}$?

Filtramos $x_1[n]$ (suma de dos tonos) con el filtro de media móvil.

---

#### 📝 Hipótesis previa

**Pregunta:** Tienes $x_1 = \sin(2\pi \cdot 0.05 \cdot n) + \sin(2\pi \cdot 0.30 \cdot n)$. Al aplicar un filtro de media móvil:

- ¿Qué seno crees que "sobrevivirá"? El de $f_1=0.05$ o el de $f_2=0.30$?
- ¿Cuál se atenuará más?

*Tu predicción:* `_______________`

---

In [ ]:
# === IMPLEMENTACIÓN ===
# TODO: Filtra x1 con el filtro de media móvil h_media
# Usa np.convolve para obtener, y1_media con la misma longitud que x1


In [ ]:
# === VALIDACIÓN ===
assert y1_media is not None, "Aplica el filtro de media móvil"
assert len(y1_media) == len(x1), "La salida debe tener la misma longitud"

print("✓ Filtrado completado")
# === VISUALIZACIÓN ===
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Solo representamos las primeras 200 muestras
nmuestras=100
n_vis = n[:nmuestras]
x1_vis = x1[:nmuestras]
x1a_vis = x1a[:nmuestras]
x1b_vis = x1b[:nmuestras]
y1_media_vis = y1_media[:nmuestras]

axes[0].plot(n_vis, x1_vis, 'b-', linewidth=1, label='Entrada $x_1$')
axes[0].plot(n_vis, x1a_vis, 'm--', linewidth=1, label='$x_{1a}$')
axes[0].plot(n_vis, x1b_vis, 'y--', linewidth=1, label='$x_{1b}$')
axes[0].set_title('ENTRADA: suma de dos tonos')
axes[0].set_ylabel('Amplitud')
axes[0].legend()

axes[1].plot(n_vis, y1_media_vis, 'r-', linewidth=1.5, label='Salida (media móvil)')
axes[1].set_title(f'SALIDA: después del filtro media móvil (M={M_media})')
axes[1].set_xlabel('n (muestras)')
axes[1].set_ylabel('Amplitud')
axes[1].legend()

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

**Describe lo que observas en 2-3 líneas:**

*Tu respuesta:*

Nota: observa fundamentalmente que señal desaparece, cual se recupera y en que circunstancias. No es necesario que entiendas el porqué, solo que describas lo que ves.

**¿Qué parámetro de `np.convolve()` has tenido que usar para obtener una salida de la misma longitud que la entrada?**

*Tu respuesta:*


---

### Bloque 1.2 · ¿Qué hace el diferenciador?

Ahora aplicamos el diferenciador: $y[n] = x[n] - x[n-1]$

---

#### 📝 Hipótesis previa

**Pregunta:** El diferenciador resalta cambios rápidos. ¿Qué seno sobrevivirá ahora?

- ¿El de $f_1=0.05$ (lento) o el de $f_2=0.30$ (rápido)?

*Tu predicción:* `_______________`

---

In [ ]:
# === IMPLEMENTACIÓN ===
# El diferenciador tiene h = [1, -1]
h_diff = np.array([1, -1])

# TODO: Filtra x1 con el diferenciador, obteniendo y1_diff con la misma longitud que x1


In [ ]:
# === VALIDACIÓN ===
assert y1_diff is not None, "Aplica el diferenciador"
print("✓ Diferenciación completada")
# === VISUALIZACIÓN COMPARATIVA ===
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

nmuestras = 100
n_vis = n[:nmuestras]

axes[0].plot(n_vis, x1[:nmuestras], 'b-', linewidth=1.5, label='$x_1 = x_{1a}+x_{1b}$')
axes[0].plot(n_vis, x1a[:nmuestras], 'm--', linewidth=1, label='$x_{1a}$ (f1)')
axes[0].plot(n_vis, x1b[:nmuestras], 'y--', linewidth=1, label='$x_{1b}$ (f2)')
axes[0].set_title('ENTRADA: $x_1$ (suma de tonos) con componentes')
axes[0].set_ylabel('Amplitud')
axes[0].legend()

axes[1].plot(n_vis, y1_media[:nmuestras], 'r-', linewidth=1.5)
axes[1].set_title(f'SALIDA media móvil (M={M_media}) ≡ filtro paso bajo')
axes[1].set_ylabel('Amplitud')

axes[2].plot(n_vis, y1_diff[:nmuestras], 'green', linewidth=1.5)
axes[2].set_title('SALIDA diferenciador ≡ filtro paso alto')
axes[2].set_xlabel('n (muestras)')
axes[2].set_ylabel('Amplitud')

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

**Compara las dos salidas en 2-3 líneas:**

*Tu respuesta:*

```
El filtro de media móvil deja pasar... El diferenciador deja pasar...
```

### Avance para talleres posteriores

> **En talleres posteriores calcularemos $|H(f)|$ de estos mismos filtros y verás *por qué* uno deja pasar frecuencias bajas y el otro frecuencias altas.**

---

## Parte 2: Implementación propia de la ecuación en diferencias

### Contexto

En la Parte 1 usaste `np.convolve` para filtros FIR sencillos. Pero la representación general de un sistema LTI es la **ecuación en diferencias**:

$$y(n) = \frac{1}{a_0}\left(\sum_{m=0}^{M} b_m\, x(n-m) \;-\; \sum_{l=1}^{L} a_l\, y(n-l)\right)$$

Esta ecuación permite representar tanto sistemas FIR como IIR. Para implementarla necesitas:

1. Un **bucle** que recorra $n = 0, 1, 2, \ldots, N-1$.
2. Tendrás que recorrer la entrada $x$ por la variable $n$, que **equivale a simular el paso del tiempo**.
3. En cada iteración, acceder a muestras pasadas de $x$ (parte FIR) y de $y$ (parte IIR).
4. **Condiciones iniciales nulas**: $x(n) = 0$ e $y(n) = 0$ para $n < 0$.

---

### Bloque 2.1 · Función `sistema()` *

Programa la función `sistema(x, b, a)` que implemente la ecuación en diferencias.

**Pistas:**
- `b` es un vector de tamaño $M+1$ y `a` es un vector de tamaño $L+1$.
- Para cada $n$, calcula primero la suma de la parte FIR ($\sum b_m \cdot x(n-m)$) y después resta la parte IIR ($\sum a_l \cdot y(n-l)$).
- Cuando $n-m < 0$ o $n-l < 0$, el valor correspondiente es 0 (condiciones iniciales nulas).

**NOTA:** La función `sistema` se desarrolla para entender mejor el funcionamiento de un sistema descrito por ecuaciones en diferencias. En el resto del taller y de la asignatura se usará `lfilter()`.

In [ ]:
def sistema(x, b, a):
    """
    Implementa la ecuación en diferencias general de un sistema LTI.
    
    Parámetros:
        x : array 1D, señal de entrada
        b : array 1D, coeficientes del numerador (parte FIR), tamaño M+1
        a : array 1D, coeficientes del denominador (parte IIR), tamaño L+1
    
    Retorna:
        y : array 1D, señal de salida (misma longitud que x)
    
    La ecuación implementada es:
        y(n) = (1/a[0]) * ( sum_{m=0}^{M} b[m]*x[n-m] - sum_{l=1}^{L} a[l]*y[n-l] )
    
    Condiciones iniciales nulas: x(n)=0, y(n)=0 para n<0.
    """

    # TODO: Calcula los valores de M y L a partir de las longitudes de b y a. Además obten N como la longitud de x.


    y = np.zeros(N)
    # TODO: Implementa el bucle que recorre n de 0 a N-1, para ir obtener cada y[n] usando la ecuación dada.
    # Recuerda manejar las condiciones iniciales nulas.

    
    return y

---

### Bloque 2.2 · Validación con el ejemplo de clase

Usamos el ejemplo IIR visto en las diapositivas de teoría:

$$y(n) - 0.8\,y(n-2) = x(n) - 0.9\,x(n-1) - x(n-2) + 0.9\,x(n-3)$$

Con la señal de prueba $x(n) = [2, -1, -3, 4, -2, 0, 5]$ y condiciones iniciales nulas.

Los coeficientes son:
- $b = [1, -0.9, -1, 0.9]$ (M=3)
- $a = [1, 0, -0.8]$ (L=2)

La salida esperada (de las diapositivas) es: $y = [2, -2.8, -2.5, 7.26, -5.5, 0.908, 6.2]$

In [ ]:
# === VALIDACIÓN CON EJEMPLO DE CLASE ===

# Señal y coeficientes del ejemplo de las diapositivas
x_ejemplo = np.array([2, -1, -3, 4, -2, 0, 5], dtype=float)
b_ejemplo = np.array([1, -0.9, -1, 0.9])
a_ejemplo = np.array([1, 0, -0.8])

# Salida esperada (de las diapositivas)
y_esperada = np.array([2, -2.8, -2.5, 7.26, -5.5, 0.908, 6.2])

# Calcula con tu función
y_sistema = sistema(x_ejemplo, b_ejemplo, a_ejemplo)

# Calcula con lfilter
y_lfilter = signal.lfilter(b_ejemplo, a_ejemplo, x_ejemplo)

# Compara
ecm_sistema_vs_lfilter = np.mean((y_sistema - y_lfilter)**2)
ecm_sistema_vs_esperada = np.mean((y_sistema - y_esperada)**2)

print(f"Salida de sistema():  y = {np.array2string(y_sistema, precision=3, separator=', ')}")
print(f"Salida de lfilter():  y = {np.array2string(y_lfilter, precision=3, separator=', ')}")
print(f"Salida esperada:      y = {np.array2string(y_esperada, precision=3, separator=', ')}")
print(f"\nError cuadrático medio (ECM) de la función sistema() vs lfilter(): {ecm_sistema_vs_lfilter:.2e}")
print(f"Error cuadrático medio (ECM) de la función sistema() vs esperada:  {ecm_sistema_vs_esperada:.2e}")

In [ ]:
# === VALIDACIÓN AUTOMÁTICA ===
assert ecm_sistema_vs_lfilter < 1e-20, f"sistema() y lfilter() difieren demasiado: ECM={ecm_sistema_vs_lfilter:.2e}"
assert ecm_sistema_vs_esperada < 1e-20, f"sistema() no coincide con la salida esperada: ECM={ecm_sistema_vs_esperada:.2e}"
print("✓ Tu función sistema() coincide con lfilter() y con la salida esperada de las diapositivas.")

---

### Bloque 2.3 · Validación con señal y sistema aleatorios

Para asegurar que tu implementación es correcta en general (no solo para el ejemplo de clase), compárala con `lfilter()` usando una señal de entrada aleatoria y coeficientes aleatorios.

In [ ]:
# === VALIDACIÓN CON SEÑAL Y COEFICIENTES ALEATORIOS ===
np.random.seed(99)

# Señal aleatoria
N_rand = 50
x_rand = np.random.randn(N_rand)

# Coeficientes aleatorios
L_rand = 3   # orden IIR
M_rand = 4   # orden FIR
b_rand = np.random.randn(M_rand + 1)
a_rand = np.random.randn(L_rand + 1)
a_rand[0] = 1.0  # normalización estándar

# Compara
y_sist_rand = sistema(x_rand, b_rand, a_rand)
y_lfilt_rand = signal.lfilter(b_rand, a_rand, x_rand)

ecm_rand = np.mean((y_sist_rand - y_lfilt_rand)**2)

print(f"Señal aleatoria de longitud {N_rand}, sistema con L={L_rand}, M={M_rand}")
print(f"Error cuadrático medio sistema() vs lfilter(): {ecm_rand:.2e}")

assert ecm_rand < 1e-10, f"sistema() y lfilter() difieren: ECM={ecm_rand:.2e}"
print("✓ Validación con señal aleatoria superada.")

### ✍️ Explicación (OBLIGATORIA)

**Preguntas:**

1. ¿La función `sistema()` devuelve el mismo resultado que `lfilter()`? ¿Cómo lo has comprobado?

   *Tu respuesta:*

2. En tu implementación de la función `sistema()` ¿Qué ocurre en las primeras iteraciones cuando $n-m < 0$ o $n-l < 0$? ¿Por qué es importante considerar esto?

   *Tu respuesta:*

3. ¿Por qué no envías a la función `sistema()` los valores de $M$ y $L$?

   *Tu respuesta:*

4. ¿Qué cambios tendrías que hacer para permitir condiciones iniciales no nulas?

   *Tu respuesta:*

### 🔍 Checkpoint del profesor (Parada 1)

- [ ] Media móvil y diferenciador aplicados a x1
- [ ] Función `sistema()` implementada con bucle explícito
- [ ] Validación con ejemplo de clase: coincide con diapositivas
- [ ] Validación con señal aleatoria: ECM ≈ 0 respecto a lfilter()

---

## Parte 3: Pruebas de señales y sistemas

### Contexto

Ahora que ya has visto como se implementa la ecuación en diferencias, usar `lfilter()`, para aplicar sistemas IIR a señales reales. En esta parte se aplicarán dos sistemas IIR (filtro paso bajo y filtro paso alto) a la señal $x_1$ (mezcla de un tono grave y otro agudo).

### Bloque 3.1 · Filtrado de tonos con sistemas IIR

Se aplicarán dos sistemas IIR a la señal $x_1$ (mezcla de un tono grave $f_1=0.05$ y otro agudo $f_2=0.30$). Los coeficientes de estos sistemas han sido diseñados para implementar:

**Filtro paso bajo** (elimina/atenúa las componentes de frecuencia altas):

$$y(n) = 0.038\,x(n) + 0.152\,x(n-1) + 0.228\,x(n-2) + 0.152\,x(n-3) + 0.038\,x(n-4)$$
$$\quad + 0.978\,y(n-1) - 0.79\,y(n-2) + 0.242\,y(n-3) - 0.038\,y(n-4)$$

**Filtro paso alto** (elimina/atenúa las componentes de frecuencia bajas):

$$y(n) = 0.038\,x(n) - 0.152\,x(n-1) + 0.228\,x(n-2) - 0.152\,x(n-3) + 0.038\,x(n-4)$$
$$\quad - 0.978\,y(n-1) - 0.79\,y(n-2) - 0.242\,y(n-3) - 0.038\,y(n-4)$$

**NOTA:** En este taller no analizaremos cómo se han obtenido estos coeficientes.

---

#### 📝 Hipótesis previa

**Pregunta:** Si aplicas el filtro paso bajo a la mezcla de un tono grave y uno agudo, ¿qué tono esperas que quede en la salida? ¿Y con el filtro paso alto?

*Tu predicción:* `_______________`

---

**Se pide:**

1. Deducir los vectores de coeficientes `b` y `a` de cada filtro a partir de las ecuaciones anteriores (recuerda la forma general de la ecuación [1]).
2. Aplicar cada filtro a $x_1$ usando `lfilter()`.
3. Observar y explicar los resultados.

In [ ]:
# === IMPLEMENTACIÓN: FILTRADO DE TONOS ===

# TODO: Define los coeficientes del filtro paso bajo, a_lpf y b_lpf


# TODO: Define los coeficientes del filtro paso alto, b_hpf y a_hpf


# TODO: Aplica los filtros a x1 usando lfilter() para obtener y1_lpf y y1_hpf


In [ ]:
# === VALIDACIÓN ===
assert y1_lpf is not None, "Aplica el filtro paso bajo"
assert y1_hpf is not None, "Aplica el filtro paso alto"
assert len(y1_lpf) == N, "La salida paso bajo debe tener la misma longitud que x1"
assert len(y1_hpf) == N, "La salida paso alto debe tener la misma longitud que x1"

# El paso bajo debe atenuar la frecuencia alta: la potencia de la salida en la
# segunda mitad del espectro debería ser menor
print(f"Potencia de x1:      {np.mean(x1**2):.4f}")
print(f"Potencia salida LPF: {np.mean(y1_lpf**2):.4f}")
print(f"Potencia salida HPF: {np.mean(y1_hpf**2):.4f}")
print("\n✓ Filtros aplicados correctamente")
# === VISUALIZACIÓN ===
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Zona estable (ignorar transitorio inicial)
n_ini = 50   # descartar transitorio inicial
n_fin = 150  # visualizar hasta esta muestra

axes[0].plot(n[n_ini:n_fin], x1[n_ini:n_fin], 'b-', linewidth=1, label='$x_1 = x_{1a}+x_{1b}$')
axes[0].plot(n[n_ini:n_fin], x1a[n_ini:n_fin], 'm--', linewidth=1, label='$x_{1a}$ (f1)')
axes[0].plot(n[n_ini:n_fin], x1b[n_ini:n_fin], 'y--', linewidth=1, label='$x_{1b}$ (f2)')
axes[0].set_title(f'ENTRADA: superposición de tonos ($f_1$={f1}, $f_2$={f2})')
axes[0].set_ylabel('Amplitud')
axes[0].legend()

axes[1].plot(n[n_ini:n_fin], y1_lpf[n_ini:n_fin], 'r-', linewidth=1.5, label='Salida LPF')
axes[1].plot(n[n_ini:n_fin], x1a[n_ini:n_fin], 'm--', linewidth=1, label='$x_{1a}$ (f1)')
axes[1].plot(n[n_ini:n_fin], x1b[n_ini:n_fin], 'y--', linewidth=1, label='$x_{1b}$ (f2)')
axes[1].set_title('SALIDA filtro PASO BAJO (IIR): debería sobrevivir el tono grave')
axes[1].set_ylabel('Amplitud')
axes[1].legend()

axes[2].plot(n[n_ini:n_fin], y1_hpf[n_ini:n_fin], 'green', linewidth=1.5, label='Salida HPF')
axes[2].plot(n[n_ini:n_fin], x1a[n_ini:n_fin], 'm--', linewidth=1, label='$x_{1a}$ (f1)')
axes[2].plot(n[n_ini:n_fin], x1b[n_ini:n_fin], 'y--', linewidth=1, label='$x_{1b}$ (f2)')
axes[2].set_title('SALIDA filtro PASO ALTO (IIR): debería sobrevivir el tono agudo')
axes[2].set_xlabel('n (muestras)')
axes[2].set_ylabel('Amplitud')
axes[2].legend()

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

1. ¿Qué señales se obtienen a la salida de cada filtro? ¿Coincide con tu hipótesis?

   *Tu respuesta:*

2. ¿Es un filtrado perfecto (se elimina completamente el otro tono) o queda algo de residuo? ¿Por qué crees que ocurre esto? ¿Existe alguna diferencia aparente con la señal que se recupera en cada caso?

   *Tu respuesta:*

3. ¿Por qué descartamos las primeras muestras en la visualización? ¿Qué es el "transitorio"?

   *Tu respuesta:*

---

## Parte 4: Filtrado por etapas (segmentación)

### Contexto

En muchas aplicaciones reales no se dispone de toda la señal de golpe (p.e., audio en tiempo real, streaming). Se necesita **procesar por trozos** (segmentos). La pregunta clave es:

> **¿Se obtiene el mismo resultado si filtramos la señal completa de una vez que si la dividimos en segmentos y filtramos cada uno por separado?**

La respuesta es: **NO**, a menos que se propaguen las **condiciones finales** de un segmento como **condiciones iniciales** del siguiente.

Esto se debe a que las posiciones de memoria del sistema (los bloques $z^{-1}$ del diagrama de bloques visto en clase) contienen valores al terminar de procesar un segmento. Si al empezar el siguiente segmento se reinician a cero, se pierde información y la salida no coincide.

---

### Bloque 4.1 · Filtrado completo vs. filtrado por segmentos

Usaremos el sistema IIR del ejemplo de clase y una señal más larga (la misma señal de prueba repetida).

La sintaxis de `lfilter` con condiciones iniciales/finales es:

```python
y, zf = signal.lfilter(b, a, x, zi=zi)
```

donde `zi` y `zf` son las condiciones iniciales y finales, respectivamente, es decir, los valores de las posiciones de memoria considerados antes y después del procesado, respectivamente. Son vectores de tamaño `max(len(a), len(b)) - 1`. Al aplicar al primer fragmento, `zi` es un vector de ceros.

In [ ]:
# === PREPARACIÓN ===

# Usamos el sistema IIR del ejemplo de clase, definido por siguientes coeficientes b_seg y a_seg.
b_seg = np.array([1, -0.9, -1, 0.9])
a_seg = np.array([1, 0, -0.8])

# Señal de entrada: a modo de ejemplo, concatenamos la misma señal de ejemplo dos veces
x_seg = np.array([2, -1, -3, 4, -2, 0, 5, 2, -1, -3, 4, -2, 0, 5], dtype=float)

# Dividimos en dos segmentos (ventana rectangular, sin solapamiento)
mitad = len(x_seg) // 2
x1_seg = x_seg[:mitad]   # primer segmento
x2_seg = x_seg[mitad:]   # segundo segmento

print(f"Señal completa x:  {x_seg}")
print(f"Segmento 1 (x1):   {x1_seg}")
print(f"Segmento 2 (x2):   {x2_seg}")

# === REFERENCIA: filtrado de la señal completa ===
y_completa = signal.lfilter(b_seg, a_seg, x_seg)
print(f"\nSalida filtrado sobre toda la señal x: y = {np.array2string(y_completa, precision=3, separator=', ')}")

---

### Bloque 4.2 · SIN considerar condiciones iniciales/finales *

Filtra cada segmento por separado con `lfilter()` sin pasar condiciones iniciales (`zi` no se usa). Concatena las salidas y compara con la salida del filtrado completo.

In [ ]:
# === SIN CONDICIONES INICIALES/FINALES ===

# Usaremos el sistema especificado por b_seg y a_seg

# TODO: Filtra el primer segmento x1_seg con lfilter (sin zi) obteniendo  y1_sinCI


# TODO: Filtra el segundo segmento x2_seg con lfilter (sin zi), obteniendo y2_sinCI


# TODO: Concatena las salidas y1_sinCI e y2_sinCI para obtener y12_sinCI, que es la salida total sin condiciones iniciales (es decir, nulas)


# TODO: Calcula el error cuadrático medio, media de las diferencias al cuadrado entre y_completa e y12_sinCI



In [ ]:
# === VALIDACIÓN ===
assert y12_sinCI is not None, "Concatena las salidas"
assert ecm_sinCI is not None, "Calcula el ECM"
assert len(y12_sinCI) == len(y_completa), "La salida concatenada debe tener la misma longitud que la salida completa"
assert ecm_sinCI > 1e-4, "El ECM es demasiado bajo, revisa que no estés usando condiciones iniciales (zi) en lfilter"

print(f"Salida filtrado completo:              y = {np.array2string(y_completa, precision=3, separator=', ')}")
print(f"Salida concatenada SIN condiciones CI:  y = {np.array2string(y12_sinCI, precision=3, separator=', ')}")
print(f"\nError cuadrático medio SIN condiciones iniciales: {ecm_sinCI:.4f}")
print(f"\n→ {'✗ Las salidas NO coinciden' if ecm_sinCI > 1e-10 else '✓ Las salidas coinciden'} ECM = {ecm_sinCI:.4f}")

---

### Bloque 4.3 · CONSIDERANDO condiciones iniciales/finales *

Ahora repite el filtrado por segmentos, pero esta vez:

1. Para el **primer segmento**, usa `zini` como vector de ceros (las condiciones iniciales son nulas). Obtén también las condiciones finales `zf`.
2. Para el **segundo segmento**, usa como `zini` las condiciones finales `zfin` del primer segmento.

**Sintaxis:** `y, zfin = signal.lfilter(b, a, x, zi=zini)`

_NOTA, en clase vimos una versión sin optimizar del sistema, pero en casos prácticos se optimizan las posiciones de memoria. En ese caso, resulta que la longitud de `zi` y `zf` es `max(len(a), len(b)) - 1`. Al arrancar, si queremos que tenga condiciones iniciales nulas, habrá que crear un vector de ceros con esa longitud._

In [ ]:
# === CON CONDICIONES INICIALES/FINALES ===

# Tamaño del vector de estado (memoria)
n_estado = max(len(a_seg), len(b_seg)) - 1

# TODO: Condiciones iniciales para el primer segmento (vector de ceros)


# TODO: Filtra el primer segmento, x1_seg, con condiciones iniciales, zini1, obteniendo la salida, y1_conCI y lascondiciones finales, zfin1


# TODO: Pasa las condiciones finales zfin1 obtenidas del primer segmento, a zini2 para el filtrado del segundo segmento


# TODO: Filtra el primer segmento, x2_seg, con condiciones iniciales, zini2, obteniendo la salida, y2_conCI y lascondiciones finales, zfin2


# TODO: Concatena las salidas y1_conCI e y2_conCI para obtener y12_conCI, que es la salida total con condiciones iniciales


# TODO: Calcula el ECM entre y_completa e y12_conCI



In [ ]:
# === VALIDACIÓN ===
assert y12_conCI is not None, "Concatena las salidas"
assert ecm_conCI is not None, "Calcula el ECM"
assert np.array_equal(zini2, zfin1), "Las condiciones finales del primer segmento deben ser las iniciales del segundo segmento"
assert len(y12_conCI) == len(y_completa), "La salida concatenada debe tener la misma longitud que la salida completa"
assert ecm_conCI < 1e-4, "El ECM es demasiado alto, revisa que estés usando las condiciones finales de la salida de un segmento como condiciones iniciales del siguiente"

print(f"Condiciones iniciales primer segmento:  zini1 = {zini1}")
print(f"Condiciones finales primer segmento:    zfin1 = {np.array2string(zfin1, precision=4)}")
print(f"Condiciones iniciales segundo segmento: zini2 = {np.array2string(zini2, precision=4)}")
print(f"Condiciones finales segundo segmento:   zfin2 = {np.array2string(zfin2, precision=4)}")
print(f"\nSalida filtrado completo:                y = {np.array2string(y_completa, precision=4, separator=', ')}")
print(f"Salida concatenada CON condiciones CI/CF: y = {np.array2string(y12_conCI, precision=4, separator=', ')}")
print(f"\nError cuadrático medio CON condiciones iniciales/finales: {ecm_conCI:.2e}")
print(f"\n→ {'✓ Las salidas COINCIDEN' if ecm_conCI < 1e-20 else '✗ Las salidas NO coinciden'} ECM = {ecm_conCI:.2e}")

In [ ]:
# === VISUALIZACIÓN COMPARATIVA ===
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

nn = np.arange(len(x_seg))

axes[0].stem(nn, y_completa, linefmt='b-', markerfmt='bo', basefmt='gray', label='Filtrado completo')
axes[0].set_title('Referencia: filtrado de la señal completa')
axes[0].set_ylabel('y(n)')
axes[0].axvline(mitad - 0.5, color='red', linestyle='--', alpha=0.5, label='División en segmentos')
axes[0].legend()

axes[1].stem(nn, y12_sinCI, linefmt='r-', markerfmt='ro', basefmt='gray', label='SIN condiciones CI/CF')
axes[1].stem(nn, y_completa, linefmt='b-', markerfmt='b+', basefmt='gray', label='Referencia')
axes[1].set_title('Filtrado por segmentos SIN propagar condiciones → NO coincide')
axes[1].set_ylabel('y(n)')
axes[1].axvline(mitad - 0.5, color='red', linestyle='--', alpha=0.5)
axes[1].legend()

axes[2].stem(nn, y12_conCI, linefmt='g-', markerfmt='go', basefmt='gray', label='CON condiciones CI/CF')
axes[2].stem(nn, y_completa, linefmt='b-', markerfmt='b+', basefmt='gray', label='Referencia')
axes[2].set_title('Filtrado por segmentos CON propagar condiciones → SÍ coincide')
axes[2].set_xlabel('n')
axes[2].set_ylabel('y(n)')
axes[2].axvline(mitad - 0.5, color='red', linestyle='--', alpha=0.5)
axes[2].legend()

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

1. ¿Por qué es necesario tener en cuenta las condiciones finales e iniciales para conectar las dos etapas del filtrado?

   *Tu respuesta:*

2. ¿Qué representan físicamente las condiciones finales `zf`? (Piensa en los bloques $z^{-1}$ del diagrama de bloques visto en clase.)

   *Tu respuesta:*


### 🔍 Checkpoint del profesor (Parada 2)

- [ ] Filtros IIR paso bajo y paso alto aplicados a x1
- [ ] Filtrado por segmentos SIN condiciones → ECM ≠ 0
- [ ] Filtrado por segmentos CON condiciones → ECM ≈ 0
- [ ] El estudiante entiende el papel de zi/zf

---

## Parte 5: Autocorrelación y correlación cruzada

### 5.1 · Autocorrelación

La **autocorrelación** mide cuánto se parece una señal a sí misma en diferentes versiones desplazadas en el tiempo. Se define como:

$$R_{xx}[m] = \sum_{n} x[n] \cdot x[n+m]$$

- **Propiedad clave:** $R_{xx}[0]$ es proporcional a la **potencia** de la señal.
- **Función en Python:** `signal.correlate(x, x, mode='full')`

---

#### 📝 Hipótesis previa

**Pregunta:** ¿Cómo esperas que sea la autocorrelación de:

a) Ruido blanco $x_2$:
- [ ] Pico estrecho solo en $m=0$ (tipo delta)
- [ ] Pico ancho centrado en $m=0$

b) Ruido suavizado $x_3$:
- [ ] Pico estrecho solo en $m=0$ (tipo delta)
- [ ] Pico ancho centrado en $m=0$

*Tu explicación:* `_______________`

---

In [ ]:
# === IMPLEMENTACIÓN ===
# TODO: Calcula la autocorrelación de x2 (ruido blanco), Rxx_blanco


# TODO: Calcula la autocorrelación de x3 (ruido suavizado), Rxx_suave



In [ ]:
# === VALIDACIÓN ===
assert Rxx_blanco is not None, "Calcula la autocorrelación del ruido blanco"
assert Rxx_suave is not None, "Calcula la autocorrelación del ruido suavizado"

# Normaliza para visualizar mejor
Rxx_blanco_norm = Rxx_blanco / np.max(Rxx_blanco)
Rxx_suave_norm = Rxx_suave / np.max(Rxx_suave)

print("✓ Autocorrelaciones calculadas")
# === VISUALIZACIÓN ===
# Vector de retardos (lags)
lags = np.arange(-N + 1, N)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Señales originales
axes[0, 0].plot(n, x2, 'orange', linewidth=0.8, alpha=0.8)
axes[0, 0].set_title('$x_2$: Ruido blanco')
axes[0, 0].set_xlabel('n')

axes[0, 1].plot(n, x3, 'green', linewidth=1)
axes[0, 1].set_title('$x_3$: Ruido suavizado')
axes[0, 1].set_xlabel('n')

# Autocorrelaciones (zoom central)
zoom = 100  # mostrar solo ±100 lags
centro = N - 1

axes[1, 0].plot(lags[centro-zoom:centro+zoom], Rxx_blanco_norm[centro-zoom:centro+zoom], 'orange', linewidth=1.5)
axes[1, 0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Autocorrelación $R_{x_2x_2}$: pico ESTRECHO (tipo delta)')
axes[1, 0].set_xlabel('Retardo m')
axes[1, 0].set_ylabel('$R_{xx}$ normalizada')

axes[1, 1].plot(lags[centro-zoom:centro+zoom], Rxx_suave_norm[centro-zoom:centro+zoom], 'green', linewidth=1.5)
axes[1, 1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Autocorrelación $R_{x_3x_3}$: pico ANCHO')
axes[1, 1].set_xlabel('Retardo m')
axes[1, 1].set_ylabel('$R_{xx}$ normalizada')

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

**Frase que debes completar:**

*Tu respuesta:*

```
"Una señal más suave tiene _______ correlación entre muestras vecinas"
```

**¿Por qué el ruido blanco tiene autocorrelación tipo delta?**

*Tu respuesta:*

```
[Piensa en la independencia estadística de las muestras]
```

---

### 5.2 · Correlación cruzada: detección de patrón oculto

La **correlación cruzada** mide el parecido entre dos señales diferentes:

$$R_{xy}[m] = \sum_{n} x[n] \cdot y[n+m]$$

- **En python:** `signal.correlate(x, y, mode='full')`
- **Aplicación clave:** Detectar si un patrón está "escondido" en una señal ruidosa.

Si insertamos un período del seno lento en medio del ruido. ¿Podremos detectarlo?

---

#### 📝 Hipótesis previa

**Pregunta:** Si insertas un patrón conocido en una señal ruidosa y calculas la correlación cruzada con ese patrón, ¿qué esperas ver?

*Tu predicción:* `_______________`

---

In [ ]:
# === PREPARACIÓN ===
# Patrón: un período del seno lento
periodo_f1 = int(1 / f1)  # período en muestras
n_patron = np.arange(periodo_f1)
patron = np.sin(2 * np.pi * f1 * n_patron)

print(f"Patrón: 1 período de seno (f={f1}), longitud={len(patron)} muestras")

# Señal: ruido con el patrón insertado en una posición
N_busq = 500
np.random.seed(123)
x_con_patron = np.random.randn(N_busq) * 0.5  # ruido de fondo

posicion_real = 200  # aquí insertamos el patrón
x_con_patron[posicion_real:posicion_real+len(patron)] += 2.0 * patron

In [ ]:
# === IMPLEMENTACIÓN ===
# TODO: Calcula la correlación cruzada entre la x_con_patron y patron



In [ ]:
# === VALIDACIÓN ===
assert correlacion is not None, "Calcula la correlación cruzada"

# Encuentra la posición del máximo
lags_corr = np.arange(-len(patron) + 1, N_busq)
idx_max = np.argmax(correlacion)
posicion_encontrada = lags_corr[idx_max]

print(f"✓ Posición real del patrón: {posicion_real}")
print(f"✓ Posición encontrada: {posicion_encontrada}")
print(f"✓ Error: {abs(posicion_real - posicion_encontrada)} muestras")
# === VISUALIZACIÓN ===
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

# Patrón
axes[0].plot(n_patron, patron, 'b-', linewidth=2)
axes[0].set_title(f'Patrón a buscar (1 período de seno, f={f1})')
axes[0].set_xlabel('n')

# Señal con patrón oculto
axes[1].plot(x_con_patron, 'gray', linewidth=0.8, alpha=0.7)
axes[1].axvspan(posicion_real, posicion_real+len(patron), color='red', alpha=0.3, label='Patrón real')
axes[1].set_title('Señal con patrón oculto en ruido')
axes[1].set_xlabel('n')
axes[1].legend()

# Correlación cruzada
axes[2].plot(lags_corr, correlacion, 'green', linewidth=1)
axes[2].axvline(posicion_encontrada, color='red', linestyle='--', linewidth=2, label=f'Máximo en n={posicion_encontrada}')
axes[2].set_title('Correlación cruzada → el pico indica la posición del patrón')
axes[2].set_xlabel('Retardo (posición)')
axes[2].legend()

plt.tight_layout()
plt.show()

### ✍️ Explicación (OBLIGATORIA)

**¿Cómo funciona la correlación cruzada para detectar el patrón?**

*Tu respuesta:*

```
[Explica qué representa el pico máximo]
```

**¿Qué pasaría si aumentamos mucho el ruido de fondo?**

*Tu respuesta:*

```
[Escribe aquí]
```

### Semilla para el Taller 9

> **La autocorrelación ancha del ruido suavizado está relacionada con que su energía se concentra en bajas frecuencias.**
>
> En el Taller 9 veremos la PSD (densidad espectral de potencia) de estas mismas señales y conectarás ambas perspectivas.
>
> La correlación cruzada es equivalente a filtrar con el patrón invertido ("filtro adaptado").

### 🔍 Checkpoint del profesor (Parada 3)

- [ ] Autocorrelación: ruido blanco → delta, ruido suavizado → ancha
- [ ] Correlación cruzada: detecta posición del patrón

---

## Preguntas de control

### P1. En la ecuación en diferencias, ¿qué papel juegan los coeficientes $a_l$ (con $l \geq 1$)? ¿Qué diferencia hay entre un sistema FIR y uno IIR?

*Tu respuesta:*

```
[Escribe aquí]
```

---

### P2. ¿Qué tipo de señales deja pasar un filtro de media móvil? ¿Y un diferenciador?

*Tu respuesta:*

```
[Escribe aquí]
```

---

### P3. ¿Por qué al segmentar una señal y filtrar cada segmento por separado (sin propagar condiciones) no se obtiene el mismo resultado que filtrar la señal completa?

*Tu respuesta:*

```
[Escribe aquí]
```

---

### P4. ¿Por qué la autocorrelación del ruido blanco es un pico estrecho en m=0?

*Tu respuesta:*

```
[Escribe aquí]
```

---

### P5. ¿Cómo usarías la correlación cruzada para sincronizar dos grabaciones de audio?

*Tu respuesta:*

```
[Escribe aquí]
```

---

## ✅ Checklist final

- [ ] Run All sin errores
- [ ] Filtro media móvil y diferenciador aplicados a suma de tonos
- [ ] Función `sistema()` implementada y validada (ejemplo de clase + aleatorio)
- [ ] Filtros IIR (paso bajo y paso alto) aplicados a señal de tonos
- [ ] Filtrado por segmentos: demostrado que SIN condiciones falla y CON condiciones funciona
- [ ] Autocorrelación de ruido blanco vs suavizado
- [ ] Correlación cruzada para detección de patrones
- [ ] Todas las explicaciones completadas
- [ ] Preguntas de control respondidas

---

## 📚 Resumen de conceptos clave

| Concepto | Descripción | Palabra clave |
|----------|-------------|---------------|
| **Ecuación en diferencias** | Representación general de un sistema LTI con coeficientes $b$ y $a$ | Sistema |
| **FIR** | Solo coeficientes $b$ (no recursivo) | Convolución finita |
| **IIR** | Coeficientes $b$ y $a$ (recursivo) | Retroalimentación |
| **`lfilter(b, a, x)`** | Función de scipy que implementa la ecuación en diferencias | Herramienta |
| **Condiciones iniciales/finales** | Estado interno del sistema entre segmentos | Memoria |
| **Filtrado por etapas** | Segmentar señal y propagar `zi`/`zf` entre segmentos | Tiempo real |
| **Autocorrelación** | Parecido de señal consigo misma desplazada | Memoria temporal |
| **Correlación cruzada** | Parecido entre dos señales | Detección |

---

## 🔗 Conexión con Taller 9

| Aquí (T6 - Tiempo) | Allí (T9 - Frecuencia) |
|-------------------|------------------------|
| Media móvil suaviza | $|H(f)|$ bajo en altas frecuencias |
| Diferenciador resalta cambios | $|H(f)|$ alto en altas frecuencias |
| Filtros IIR paso bajo/alto | Respuesta en frecuencia con polos y ceros |
| Autocorrelación ancha = suave | PSD concentrada en bajas frecuencias |
| Autocorrelación estrecha = ruido | PSD plana (espectro blanco) |

**En el Taller 9 usarás exactamente las mismas señales $x_1, x_2, x_3$ y los mismos filtros, pero calcularás espectros y respuestas en frecuencia para justificar lo que aquí has observado.**

---

**Fin del Taller 6**